# 01 - Video to Cropped Frames

## Purpose
Convert incoming Copernicus Browser MP4 videos and timeline TXT files into the project dataset structure.

## Inputs
- `raw_to_be_processed/<latitude>_<longitude>_<label>.mp4`
- `raw_to_be_processed/<latitude>_<longitude>.txt`
- Optional fallback TXT name: `<latitude>_<longitude>_<label>.txt`

## Outputs
- Raw archived files under `data/raw/<label>/<sample_id>/`
- Cropped PNG frames under `data/processed/<label>/<sample_id>/`
- Per-sample frame metadata
- Updated `data/processed/sample_index.csv`

## Notes
This notebook only handles video/frame preprocessing. It does not call Google Earth Engine and does not engineer GEE features.

## 1. Configure Paths and Runtime Options
Defines the project root, input folder, output data folder, crop percentage, and whether existing frame outputs should be regenerated.

In [ ]:
from pathlib import Path
import sys
import importlib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INBOX_DIR = PROJECT_ROOT / "raw_to_be_processed"
DATA_DIR = PROJECT_ROOT / "data"
CROP_PERCENT = 5.0
FORCE_REPROCESS = False

## 2. Load Preprocessing Helpers
Reloads the raw-video preprocessing module so notebook reruns pick up local code edits without restarting the kernel.

In [ ]:
import src.preprocessing.process_raw_videos as process_raw_videos
importlib.reload(process_raw_videos)

process_inbox = process_raw_videos.process_inbox
write_sample_index = process_raw_videos.write_sample_index

## 3. Process Inbox and Rebuild Sample Index
Processes any MP4 files found in `raw_to_be_processed`. If no new MP4 files exist, it only rebuilds `sample_index.csv` from the current processed folders.

In [ ]:
if INBOX_DIR.exists() and any(INBOX_DIR.glob("*.mp4")):
    results = process_inbox(
        inbox_dir=INBOX_DIR,
        data_dir=DATA_DIR,
        crop_percent=CROP_PERCENT,
        force=FORCE_REPROCESS,
    )
    print(f"Processed/skipped samples: {len(results)}")
else:
    print(f"No MP4 files found in {INBOX_DIR}; rebuilding sample index only.")

sample_index_path = write_sample_index(DATA_DIR)
print(f"Sample index: {sample_index_path.relative_to(PROJECT_ROOT)}")

## 4. Inspect Sample Index
Loads the generated `sample_index.csv` to verify the number of samples and preview the dataset path contract.

In [ ]:
import pandas as pd

sample_index = pd.read_csv(DATA_DIR / "processed" / "sample_index.csv")
print(sample_index.shape)
sample_index.head()